In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [17]:
df = pd.read_csv("Titanic-Dataset.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Inspecting the dataset

In [18]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

Shape: (891, 12)

Columns:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [19]:
data = df.copy()

# Extracting Title from Name

In [20]:
data["Title"] = data["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
data[["Name", "Title"]].head()

,Name,Title
0,"Braund, Mr. Owen Harris",Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,"Heikkinen, Miss. Laina",Miss
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs
4,"Allen, Mr. William Henry",Mr


# Grouping rare titles together

In [21]:
data["Title"] = data["Title"].replace([
    "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major",
    "Rev", "Sir", "Jonkheer", "Dona"
], "Rare")

data["Title"] = data["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

print(data["Title"].value_counts())

Title
Mr              517
Miss            185
Mrs             126
Master           40
Rare             22
the Countess      1
Name: count, dtype: int64


# Filling missing values in Embarked and Age

In [23]:
data["Embarked"] = data["Embarked"].fillna(data["Embarked"].mode()[0])

data["Age"] = data["Age"].fillna(data.groupby(["Title", "Pclass"])["Age"].transform("median"))

# If any Age values are still missing, fill with overall median
data["Age"] = data["Age"].fillna(data["Age"].median())

# Dropping unnecessary columns

In [24]:
data = data.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,0,3,male,22.0,1,0,7.2500,S,Mr
1,1,1,female,38.0,1,0,71.2833,C,Mrs
2,1,3,female,26.0,0,0,7.9250,S,Miss
3,1,1,female,35.0,1,0,53.1000,S,Mrs
4,0,3,male,35.0,0,0,8.0500,S,Mr


# Encoding Sex as binary

In [25]:
# male = 0
# female = 1
data["Sex"] = data["Sex"].map({"male": 0, "female": 1})
data["Sex"].head()

0    0
1    1
2    1
3    1
4    0
Name: Sex, dtype: int64

# One-hot encoding categorical columns (Embarked & Title)

In [26]:
data = pd.get_dummies(data, columns=["Embarked", "Title"], drop_first=True)
data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Title_the Countess
0,0,3,0,22.0,1,0,7.2500,False,True,False,True,False,False,False
1,1,1,1,38.0,1,0,71.2833,False,False,False,False,True,False,False
2,1,3,1,26.0,0,0,7.9250,False,True,True,False,False,False,False
3,1,1,1,35.0,1,0,53.1000,False,True,False,False,True,False,False
4,0,3,0,35.0,0,0,8.0500,False,True,False,True,False,False,False


# Checking final missing values

In [27]:
print("Missing values after preprocessing:")
print(data.isnull().sum())

print("\nFinal shape after preprocessing:", data.shape)

Missing values after preprocessing:
Survived              0
Pclass                0
Sex                   0
Age                   0
SibSp                 0
Parch                 0
Fare                  0
Embarked_Q            0
Embarked_S            0
Title_Miss            0
Title_Mr              0
Title_Mrs             0
Title_Rare            0
Title_the Countess    0
dtype: int64

Final shape after preprocessing: (891, 14)


# Spliting into features and target

In [28]:
X = data.drop("Survived", axis=1)
y = data["Survived"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (891, 13)
y shape: (891,)


# Train / validation / test split

In [29]:
# 70% Training, 15% Validation and 15% Testing
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

Training set: (623, 13) (623,)
Validation set: (134, 13) (134,)
Test set: (134, 13) (134,)


# Scaling numerical features (Age and Fare)

In [30]:
scaler = StandardScaler()

numerical_cols = ["Age", "Fare"]

X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_val[numerical_cols] = scaler.transform(X_val[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# Viewing final processed datasets

In [33]:
print("X_train preview:")
display(X_train.head())

print("y_train preview:")
display(y_train.head())

X_train preview:


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Title_the Countess
748,1,0,-0.771085,1,0,0.465738,False,True,False,True,False,False,False
45,3,0,-0.253581,0,0,-0.478269,False,True,False,True,False,False,False
28,3,1,-0.845015,0,0,-0.481848,True,False,True,False,False,False,False
633,1,0,0.781428,0,0,-0.646954,False,True,False,True,False,False,False
403,3,0,-0.105722,1,0,-0.314823,False,True,False,True,False,False,False


y_train preview:


748    0
45     0
28     1
633    0
403    0
Name: Survived, dtype: int64

# Final checks

In [32]:
print("Training missing values:\n", X_train.isnull().sum())
print("\nValidation missing values:\n", X_val.isnull().sum())
print("\nTest missing values:\n", X_test.isnull().sum())

Training missing values:
 Pclass                0
Sex                   0
Age                   0
SibSp                 0
Parch                 0
Fare                  0
Embarked_Q            0
Embarked_S            0
Title_Miss            0
Title_Mr              0
Title_Mrs             0
Title_Rare            0
Title_the Countess    0
dtype: int64

Validation missing values:
 Pclass                0
Sex                   0
Age                   0
SibSp                 0
Parch                 0
Fare                  0
Embarked_Q            0
Embarked_S            0
Title_Miss            0
Title_Mr              0
Title_Mrs             0
Title_Rare            0
Title_the Countess    0
dtype: int64

Test missing values:
 Pclass                0
Sex                   0
Age                   0
SibSp                 0
Parch                 0
Fare                  0
Embarked_Q            0
Embarked_S            0
Title_Miss            0
Title_Mr              0
Title_Mrs             0
Title_Ra